In [7]:
import numpy as np
import matplotlib.pyplot as plt
from lm_polygraph.ue_metrics.pred_rej_area import PredictionRejectionArea
from lm_polygraph.ue_metrics.ue_metric import (
    get_random_scores,
    normalize_metric,
)
import sklearn
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
from collections import defaultdict
from sacrebleu import CHRF, BLEU
from lm_polygraph.utils.manager import UEManager

from lm_polygraph.ue_metrics import PredictionRejectionArea

metrics = [PredictionRejectionArea(max_rejection=0.5)]

In [8]:
def load_managers(dataset, model='llama', model_type='base', suff ='test_qe_enriched'):
    prefix = '' if model_type == 'base' else '_instruct'

    train_manager =  UEManager.load(f'processed_mans/{model}{prefix}_{dataset}_train_{suff}.man') if suff!='' else  UEManager.load(f'mans/{model}{prefix}_{dataset}_train.man') 

    return train_manager

def extract_and_prepare_data_train(dataset, methods_dict, all_metrics, model='llama', model_type='base', suff ='test_qe_enriched'):
    train_manager = load_managers(dataset, model, model_type, suff=suff)

    full_ue_methods = list(methods_dict.keys())
    ue_methods = list(methods_dict.values())

    
    train_sequences = train_manager.stats['greedy_tokens']
    train_texts = train_manager.stats['greedy_texts']
    train_targets = train_manager.stats['target_texts']

    train_gen_lengths = np.array([len(seq) for seq in train_sequences])
    # gen_lengths = np.array([len(seq) for seq in sequences])

    # Get train and test values for metrics and UE, remove union of nans
    test_nans = []
    train_nans = []

    train_metric_values = {}
    test_metric_values = {}
    for metric in all_metrics:
        # values = np.array(manager.gen_metrics[('sequence', metric)])
        # test_metric_values[metric] = np.array(values)
        # test_nans.extend(np.argwhere(np.isnan(values)).flatten())

        train_values = np.array(train_manager.gen_metrics[('sequence', metric)])
        train_metric_values[metric] = np.array(train_values)
        train_nans.extend(np.argwhere(np.isnan(train_values)).flatten())

    train_ue_values = {}
    # test_ue_values = {}
    for i, method in enumerate(full_ue_methods):
        train_values = np.array(train_manager.estimations[('sequence', method)])
        train_ue_values[ue_methods[i]] = train_values
        train_nans.extend(np.argwhere(np.isnan(train_values)).flatten())

        # values = np.array(manager.estimations[('sequence', method)])
        # test_ue_values[ue_methods[i]] = values
        # test_nans.extend(np.argwhere(np.isnan(values)).flatten())

    train_nans = np.unique(train_nans).astype(int)
    # test_nans = np.unique(test_nans).astype(int)

    # Remove nans
    for metric in all_metrics:
        # test_metric_values[metric] = np.delete(test_metric_values[metric], test_nans)
        train_metric_values[metric] = np.delete(train_metric_values[metric], train_nans)

    for method in ue_methods:
        # test_ue_values[method] = np.delete(test_ue_values[method], test_nans)
        train_ue_values[method] = np.delete(train_ue_values[method], train_nans)

    train_gen_lengths = np.delete(train_gen_lengths, train_nans)

    return train_ue_values, train_metric_values, train_gen_lengths


In [21]:
methods_dict = {
    'MaximumSequenceProbability': 'MSP',
    'Perplexity': 'PPL',
    'MeanTokenEntropy': 'MTE',
    'MonteCarloSequenceEntropy': 'MCSE',
    'MonteCarloNormalizedSequenceEntropy': 'MCNSE',
}

DATASETS = [
    'wmt14_deen',
    'wmt14_fren',
    'wmt14_csen',
    'wmt14_ruen',
    'wmt19_ruen',
    'wmt19_fien',
    'wmt19_deen',
    'wmt19_lten'
]

all_metrics = ['Comet-wmt22-comet-da', 'XComet-XCOMET-XXL', 'metricx-metricx-24-hybrid-large-v2p6']
all_methods =['MSP', 'PPL', 'MTE', 'MCSE', 'MCNSE']

In [22]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from scipy.stats import linregress
import numpy as np
import pandas as pd

sns.set(style="whitegrid", font_scale=1.4, rc={"font.family": "serif"})

methrics_dict ={
  'Comet':'Comet', 'XComet-XCOMET-XXL' :'XComet-XXL', 'metricx-metricx-24-hybrid-large-v2p6' :'MetricX-Large' ,
  'AlignScoreInputOutput':'Align Score', 'Accuracy':'Acc', 'AlignScoreInputOutput':'Align Score','Rouge_rougeL':'Rouge L'
}

def format_dataset_name(raw_name):
    try:
        prefix, lang_pair = raw_name.split("_")
        prefix = prefix.upper()

        if len(lang_pair) == 4:  # e.g., fren → Fr-En
            src = lang_pair[:2].capitalize()
            tgt = lang_pair[2:].capitalize()
            lang_fmt = f"{src}-{tgt}"
        else:
            lang_fmt = lang_pair.upper()

        return f"{prefix} {lang_fmt}"
    except Exception:
        return raw_name.upper()


def plot_metric_vs_length(
    gen_lengths, metric_values,
    metric_name, dataset_name,
    mode='Train', save_path='plot.pdf', model=''
):
    from sklearn.linear_model import LinearRegression
    from scipy.stats import linregress

    # Trim outliers
    upper_q, lower_q = np.quantile(gen_lengths, [0.95, 0.05])
    mask = (gen_lengths > lower_q) & (gen_lengths < upper_q)
    gen_lengths = gen_lengths[mask]
    metric_values = metric_values[mask]

    # Normalize
    scaler_len = MinMaxScaler()
    scaler_val = MinMaxScaler()

    norm_len = scaler_len.fit_transform(gen_lengths[:, None]).squeeze()
    norm_val = scaler_val.fit_transform(metric_values[:, None]).squeeze()

    # Bin and smooth
    df = pd.DataFrame({"length": norm_len, "metric": norm_val})
    grouped = df.groupby("length").agg(['mean', 'sem'])
    x_vals = grouped.index.values
    y_vals = grouped['metric']['mean'].values
    y_errs = grouped['metric']['sem'].values

    # Fit regression (on raw normalized data)
    linreg = LinearRegression().fit(norm_len[:, None], norm_val)
    slope = linreg.coef_[0]

    # Also compute p-value
    slope_, intercept_, r_val, p_val, std_err = linregress(norm_len, norm_val)

    x_line = np.linspace(0, 1, 100)
    y_line = linreg.predict(x_line[:, None])

    return slope, p_val


In [ ]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np
import os 
# DATASETS =['xsum']
# all_metrics = ['AlignScoreInputOutput', 'Rouge_rougeL']
models =['llama','gemma', 'eurollm']
records=[]
for model in models:

    for dataset in DATASETS:
        train_ue_values, train_metric_values, train_gen_lengths = extract_and_prepare_data_train(dataset, methods_dict, all_metrics, model=model, suff='full_enriched')

        for metric in all_metrics:
            os.makedirs(f'{model}/{metric}', exist_ok=True)

            slope, p_val = plot_metric_vs_length(
                gen_lengths=np.array(train_gen_lengths),
                metric_values=np.array(train_metric_values[metric]),
                metric_name=metric,
                dataset_name=dataset,
                mode='Train',
                save_path=f'{model}/{metric}/{dataset}_{metric}_{model}_train.pdf',
                # model = model
            )
            records.append({'model':model, 'metric':metric, 'dataset':dataset, 'slope': slope, 'p_val':p_val })
    
df = pd.DataFrame(records)

# Save to CSV
df.to_csv("nmt_values.csv", index=False)


In [ ]:
methods_dict = {
    'MaximumSequenceProbability': 'MSP',
    'Perplexity': 'PPL',
    'MeanTokenEntropy': 'MTE',
    'MonteCarloSequenceEntropy': 'MCSE',
    'MonteCarloNormalizedSequenceEntropy': 'MCNSE',
        'LexicalSimilarity_rougeL': 'LSRL'
}

methrics_dict ={
  'MSP':'MSP', 'PPL' :'PPL', 'MTE' :'MTE',  'MCSE':'MCSE', 'MCNSE':'MCNSE', 'LSRL': 'LSRL'
}

DATASETS = [
    'wmt14_deen',
    'wmt14_fren',
    'wmt14_csen',
    'wmt14_ruen',
    'wmt19_ruen',
    'wmt19_fien',
    'wmt19_deen',
    'wmt19_lten'
]

all_metrics = ['Comet-wmt22-comet-da', 'XComet-XCOMET-XXL', 'metricx-metricx-24-hybrid-large-v2p6']
models = ['llama', 'gemma', 'eurollm']
records =[]
for model in models:
    for dataset in DATASETS:
        train_ue_values, train_metric_values, train_gen_lengths = extract_and_prepare_data_train(dataset, methods_dict, all_metrics, model=model, suff='full_enriched')

        for metric, metric_short  in methods_dict.items():
            # os.makedirs(f'{model}/{metric}', exist_ok=True)
            slope, p_val = plot_metric_vs_length(
                gen_lengths=np.array(train_gen_lengths),
                metric_values=np.array(train_ue_values[metric_short]),
                metric_name=metric_short,
                dataset_name=dataset,
                mode='Train',
                save_path=f'{model}/{metric}/{dataset}_{metric}_{model}_train.pdf',
                # model = model
            )
            records.append({'model':model, 'metric':metric_short, 'dataset':dataset, 'slope': slope, 'p_val':p_val })

df = pd.DataFrame(records)
df.to_csv("nmt_ue_values.csv", index=False)


In [ ]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np
import os 
DATASETS =['xsum']
all_metrics = ['AlignScoreInputOutput']
models =['llama','gemma']
records=[]
for model in models:

    for dataset in DATASETS:
        train_ue_values, train_metric_values, train_gen_lengths = extract_and_prepare_data_train(dataset, methods_dict, all_metrics, model=model, suff='full_enriched')

        for metric in all_metrics:
            os.makedirs(f'{model}/{metric}', exist_ok=True)

            slope, p_val = plot_metric_vs_length(
                gen_lengths=np.array(train_gen_lengths),
                metric_values=np.array(train_metric_values[metric]),
                metric_name=metric,
                dataset_name=dataset,
                mode='Train',
                save_path=f'{model}/{metric}/{dataset}_{metric}_{model}_train.pdf',
                # model = model
            )
            records.append({'model':model, 'metric':metric, 'dataset':dataset, 'slope': slope, 'p_val':p_val })
    
df = pd.DataFrame(records)

# Save to CSV
df.to_csv("sum_values.csv", index=False)


In [ ]:
methods_dict = {
    'MaximumSequenceProbability': 'MSP',
    'Perplexity': 'PPL',
    'MeanTokenEntropy': 'MTE',
    'MonteCarloSequenceEntropy': 'MCSE',
    'MonteCarloNormalizedSequenceEntropy': 'MCNSE',
        'LexicalSimilarity_rougeL': 'LSRL'
}

methrics_dict ={
  'MSP':'MSP', 'PPL' :'PPL', 'MTE' :'MTE',  'MCSE':'MCSE', 'MCNSE':'MCNSE', 'LSRL': 'LSRL'
}

DATASETS =['xsum']
all_metrics = ['AlignScoreInputOutput']
models =['llama','gemma']
records =[]
for model in models:
    for dataset in DATASETS:
        train_ue_values, train_metric_values, train_gen_lengths = extract_and_prepare_data_train(dataset, methods_dict, all_metrics, model=model, suff='full_enriched')

        for metric, metric_short  in methods_dict.items():
            # os.makedirs(f'{model}/{metric}', exist_ok=True)
            slope, p_val = plot_metric_vs_length(
                gen_lengths=np.array(train_gen_lengths),
                metric_values=np.array(train_ue_values[metric_short]),
                metric_name=metric_short,
                dataset_name=dataset,
                mode='Train',
                save_path=f'{model}/{metric}/{dataset}_{metric}_{model}_train.pdf',
                # model = model
            )
            records.append({'model':model, 'metric':metric_short, 'dataset':dataset, 'slope': slope, 'p_val':p_val })

df = pd.DataFrame(records)
df.to_csv("sum_ue_values.csv", index=False)


In [ ]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np
import os 
DATASETS =['gsm8k']
all_metrics = ['Accuracy']
models =['llama','gemma']
records=[]
for model in models:

    for dataset in DATASETS:
        train_ue_values, train_metric_values, train_gen_lengths = extract_and_prepare_data_train(dataset, methods_dict, all_metrics, model=model, suff='full_enriched')

        for metric in all_metrics:
            os.makedirs(f'{model}/{metric}', exist_ok=True)

            slope, p_val = plot_metric_vs_length(
                gen_lengths=np.array(train_gen_lengths),
                metric_values=np.array(train_metric_values[metric]),
                metric_name=metric,
                dataset_name=dataset,
                mode='Train',
                save_path=f'{model}/{metric}/{dataset}_{metric}_{model}_train.pdf',
                # model = model
            )
            records.append({'model':model, 'metric':metric, 'dataset':dataset, 'slope': slope, 'p_val':p_val })
    
df = pd.DataFrame(records)

# Save to CSV
df.to_csv("qa_values.csv", index=False)


In [53]:
methods_dict = {
    'MaximumSequenceProbability': 'MSP',
    'Perplexity': 'PPL',
    'MeanTokenEntropy': 'MTE',
    'MonteCarloSequenceEntropy': 'MCSE',
    'MonteCarloNormalizedSequenceEntropy': 'MCNSE',
        'LexicalSimilarity_rougeL': 'LSRL'
}

methrics_dict ={
  'MSP':'MSP', 'PPL' :'PPL', 'MTE' :'MTE',  'MCSE':'MCSE', 'MCNSE':'MCNSE', 'LSRL': 'LSRL'
}

DATASETS =['gsm8k']
all_metrics = ['Accuracy']
models =['llama','gemma']
records =[]
for model in models:
    for dataset in DATASETS:
        train_ue_values, train_metric_values, train_gen_lengths = extract_and_prepare_data_train(dataset, methods_dict, all_metrics, model=model, suff='full_enriched')

        for metric, metric_short  in methods_dict.items():
            # os.makedirs(f'{model}/{metric}', exist_ok=True)
            slope, p_val = plot_metric_vs_length(
                gen_lengths=np.array(train_gen_lengths),
                metric_values=np.array(train_ue_values[metric_short]),
                metric_name=metric_short,
                dataset_name=dataset,
                mode='Train',
                save_path=f'{model}/{metric}/{dataset}_{metric}_{model}_train.pdf',
                # model = model
            )
            records.append({'model':model, 'metric':metric_short, 'dataset':dataset, 'slope': slope, 'p_val':p_val })

df = pd.DataFrame(records)
df.to_csv("qa_ue_values.csv", index=False)


Loading NLI model...
/home/maiya.goloburda/.conda/envs/detrend/lib/python3.10/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
Some weights of the model checkpoint at microsoft/deberta-large-mnli were not used when initializing DebertaForSequenceClassification: ['config']
- This IS expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Initializing stat calculators...
Initializing InitialStateCalculator
Initializing SemanticMatrixCalculator
Initializing SemanticClassesCalculator
Initializing 

Stat calculators: [<lm_polygraph.stat_calculators.greedy_probs.GreedyProbsCalculator object at 0x7f72b5303700>]


Some weights of the model checkpoint at microsoft/deberta-large-mnli were not used when initializing DebertaForSequenceClassification: ['config']
- This IS expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Initializing stat calculators...
Initializing InitialStateCalculator
Initializing SemanticMatrixCalculator
Initializing SemanticClassesCalculator
Initializing GreedyProbsCalculator
Initializing EntropyCalculator
Initializing GreedyLMProbsCalculator
Initializing SamplingGenerationCalculator
Initializing BartScoreCalculator
Initializing ModelScoreCalculat

Stat calculators: [<lm_polygraph.stat_calculators.greedy_probs.GreedyProbsCalculator object at 0x7f72b5c9bb20>]


In [32]:
import pandas as pd 

# df1 =pd.read_csv('qa_values.csv')
# df2 =pd.read_csv('sum_values.csv')
df3 =pd.read_csv('nmt_values.csv')
# df1,df2,
df = pd.concat([df3])
all_metrics = ['Comet-wmt22-comet-da', 'XComet-XCOMET-XXL', 'metricx-metricx-24-hybrid-large-v2p6']

df.tail()

,model,metric,dataset,slope,p_val
67,eurollm,XComet-XCOMET-XXL,wmt19_deen,-0.095519,1.366746e-07
68,eurollm,metricx-metricx-24-hybrid-large-v2p6,wmt19_deen,-0.050112,1.532200e-04
69,eurollm,Comet-wmt22-comet-da,wmt19_lten,-0.026960,5.332963e-02
70,eurollm,XComet-XCOMET-XXL,wmt19_lten,0.156026,5.616706e-12
71,eurollm,metricx-metricx-24-hybrid-large-v2p6,wmt19_lten,0.026060,7.761378e-02


In [ ]:
import pandas as pd

# Assuming df is already loaded as in your screenshot
blocks = []
for model in df['model'].unique():
    model_df = df[df['model'] == model].copy()
    model_df = model_df.drop(columns='model')
    model_df.insert(0, '', ['\\rowcolor[gray]{0.9} \\textbf{' + model + '}'] + [''] * (len(model_df) - 1))
    blocks.append(model_df)

final_df = pd.concat(blocks)

# Convert to LaTeX
latex_table = final_df.to_latex(index=False, escape=False, column_format='lllll', float_format="%.5f")
print(latex_table)


In [35]:
import pandas as pd

# Assume df is your input DataFrame
# Columns: model, dataset, metric, slope, p_val

# Optional renaming
metric_map = {
    "Comet-wmt22-comet-da": "COMET",
    "XComet-XCOMET-XXL": "XCOMET",
    "metricx-metricx-24-hybrid-large-v2p6": "MetricX"
}
df['metric'] = df['metric'].map(metric_map).fillna(df['metric'])

# Pivot to wide format with slope and p_val per metric
pivot = df.pivot_table(
    index=['model', 'dataset'],
    columns='metric',
    values=['slope', 'p_val'],
    aggfunc='first'
)

# Flatten MultiIndex columns
pivot.columns = [f"{stat}_{metric}" for stat, metric in pivot.columns]
pivot = pivot.reset_index()

# Define the metric order
metrics = ['COMET', 'MetricX', 'XCOMET']
columns = ['model', 'dataset']
for m in metrics:
    columns += [f'slope_{m}', f'p_val_{m}']
pivot = pivot.reindex(columns=columns)

# Build LaTeX table
lines = []

# Table header
lines.append("\\begin{tabular}{ll" + "cc" * len(metrics) + "}")
lines.append("\\toprule")
lines.append("\\textbf{Model} & \\textbf{Dataset} & " + " & ".join([f"\\multicolumn{{2}}{{c}}{{{m}}}" for m in metrics]) + " \\\\")
lines.append(" & & " + " & ".join(["\\textit{slope} & \\textit{p-val}"] * len(metrics)) + " \\\\")
lines.append("\\midrule")

# Model blocks
for model, group in pivot.groupby("model"):
    lines.append(f"\\rowcolor[gray]{{0.9}} \\multicolumn{{{2 + 2 * len(metrics)}}}{{l}}{{\\textbf{{{model}}}}} \\\\")
    for _, row in group.iterrows():
        row_str = f"{row['model']} & {row['dataset']}"
        for m in metrics:
            s = row.get(f'slope_{m}', '')
            p = row.get(f'p_val_{m}', '')
            s_str = f"{s:.3f}" if pd.notna(s) else ""
            p_str = f"{p:.3f}" if pd.notna(p) else ""
            row_str += f" & {s_str} & {p_str}"
        lines.append(row_str + " \\\\")

lines.append("\\bottomrule")
lines.append("\\end{tabular}")

# Final LaTeX string
latex_code = "\n".join(lines)
print(latex_code)


\begin{tabular}{llcccccc}
\toprule
\textbf{Model} & \textbf{Dataset} & \multicolumn{2}{c}{COMET} & \multicolumn{2}{c}{MetricX} & \multicolumn{2}{c}{XCOMET} \\
 & & \textit{slope} & \textit{p-val} & \textit{slope} & \textit{p-val} & \textit{slope} & \textit{p-val} \\
\midrule
\rowcolor[gray]{0.9} \multicolumn{8}{l}{\textbf{eurollm}} \\
eurollm & wmt14_csen & -0.047 & 0.001 & 0.027 & 0.040 & -0.011 & 0.572 \\
eurollm & wmt14_deen & -0.016 & 0.413 & 0.002 & 0.929 & -0.035 & 0.157 \\
eurollm & wmt14_fren & -0.029 & 0.042 & 0.007 & 0.666 & -0.034 & 0.152 \\
eurollm & wmt14_ruen & -0.101 & 0.000 & -0.058 & 0.001 & -0.031 & 0.250 \\
eurollm & wmt19_deen & -0.156 & 0.000 & -0.050 & 0.000 & -0.096 & 0.000 \\
eurollm & wmt19_fien & 0.007 & 0.648 & 0.104 & 0.000 & 0.170 & 0.000 \\
eurollm & wmt19_lten & -0.027 & 0.053 & 0.026 & 0.078 & 0.156 & 0.000 \\
eurollm & wmt19_ruen & -0.034 & 0.003 & 0.049 & 0.000 & 0.135 & 0.000 \\
\rowcolor[gray]{0.9} \multicolumn{8}{l}{\textbf{gemma}} \\
gemma & wmt14_

In [48]:
df = pd.read_csv('nmt_ue_values.csv')

df

,model,metric,dataset,slope,p_val
0,llama,MSP,wmt14_deen,-0.190048,0.000026
1,llama,PPL,wmt14_deen,-0.190048,0.000026
2,llama,MTE,wmt14_deen,-0.190048,0.000026
3,llama,MCSE,wmt14_deen,-0.190048,0.000026
4,llama,MCNSE,wmt14_deen,-0.190048,0.000026
...,...,...,...,...,...
139,eurollm,PPL,wmt19_lten,-0.190048,0.000026
140,eurollm,MTE,wmt19_lten,-0.190048,0.000026
141,eurollm,MCSE,wmt19_lten,-0.190048,0.000026
142,eurollm,MCNSE,wmt19_lten,-0.190048,0.000026


In [51]:
import pandas as pd

# Assume df is your input DataFrame with columns: model, dataset, metric, slope, p_val

# Optional renaming
df = pd.read_csv('nmt_ue_values.csv')
metric_map = {
    # "Comet-wmt22-comet-da": "COMET",
    # "XComet-XCOMET-XXL": "XCOMET",
    # "metricx-metricx-24-hybrid-large-v2p6": "MetricX"
      'MSP':'MSP', 'PPL' :'PPL', 'MTE' :'MTE',  'MCSE':'MCSE', 'MCNSE':'MCNSE', 'LSRL': 'LSRL'
}
df['metric'] = df['metric'].map(metric_map).fillna(df['metric'])

# Pivot to wide format with slope and p_val per metric
pivot = df.pivot_table(
    index=['model', 'dataset'],
    columns='metric',
    values=['slope', 'p_val'],
    aggfunc='first'
)

# Flatten MultiIndex columns
pivot.columns = [f"{stat}_{metric}" for stat, metric in pivot.columns]
pivot = pivot.reset_index()

# Define metric order
# metrics = ['COMET', 'MetricX', 'XCOMET']
metrics =['MSP', 'PPL', 'MTE',  'MCSE','MCNSE',  'LSRL']
columns = ['model', 'dataset']
for m in metrics:
    columns += [f'slope_{m}', f'p_val_{m}']
pivot = pivot.reindex(columns=columns)

# Build LaTeX table
lines = []

# Table header
lines.append("\\begin{tabular}{l" + "cc" * len(metrics) + "}")
lines.append("\\toprule")
lines.append("\\textbf{Dataset} & " + " & ".join([f"\\multicolumn{{2}}{{c}}{{{m}}}" for m in metrics]) + " \\\\")
lines.append(" & " + " & ".join(["slope & p-val"] * len(metrics)) + " \\\\")
lines.append("\\midrule")

# Model blocks without repeating model name in rows
for model, group in pivot.groupby("model"):
    lines.append("\\midrule")
    lines.append(f"\\multicolumn{{{1 + 2 * len(metrics)}}}{{c}}{{\\textbf{{{model}}}}} \\\\")
    lines.append("\\midrule")
    for _, row in group.iterrows():
        row_str = f"{format_dataset_name(row['dataset'])}"
        for m in metrics:
            s = row.get(f'slope_{m}', '')
            p = row.get(f'p_val_{m}', '')
            s_str = f"{s:.3f}" if pd.notna(s) else ""
            p_str = f"{p:.3f}" if pd.notna(p) else ""
            row_str += f" & {s_str} & {p_str}"
        lines.append(row_str + " \\\\")

lines.append("\\bottomrule")
lines.append("\\end{tabular}")

# Final LaTeX output
latex_code = "\n".join(lines)
print(latex_code)


\begin{tabular}{lcccccccccccc}
\toprule
\textbf{Dataset} & \multicolumn{2}{c}{MSP} & \multicolumn{2}{c}{PPL} & \multicolumn{2}{c}{MTE} & \multicolumn{2}{c}{MCSE} & \multicolumn{2}{c}{MCNSE} & \multicolumn{2}{c}{LSRL} \\
 & slope & p-val & slope & p-val & slope & p-val & slope & p-val & slope & p-val & slope & p-val \\
\midrule
\midrule
\multicolumn{13}{c}{\textbf{eurollm}} \\
\midrule
WMT14 Cs-En & 0.299 & 0.000 & -0.056 & 0.000 & -0.091 & 0.000 & 0.424 & 0.000 & -0.057 & 0.000 & -0.014 & 0.426 \\
WMT14 De-En & 0.359 & 0.000 & -0.083 & 0.000 & -0.117 & 0.000 & 0.340 & 0.000 & -0.025 & 0.109 & 0.011 & 0.553 \\
WMT14 Fr-En & 0.368 & 0.000 & -0.121 & 0.000 & -0.141 & 0.000 & 0.169 & 0.000 & -0.042 & 0.002 & 0.000 & 0.988 \\
WMT14 Ru-En & 0.266 & 0.000 & -0.136 & 0.000 & -0.195 & 0.000 & 0.227 & 0.000 & -0.171 & 0.000 & -0.164 & 0.000 \\
WMT19 De-En & 0.203 & 0.000 & -0.043 & 0.002 & -0.106 & 0.000 & 0.265 & 0.000 & -0.071 & 0.000 & -0.015 & 0.433 \\
WMT19 Fi-En & 0.462 & 0.000 & -0.037 & 

In [55]:
import pandas as pd

# Assume df is your input DataFrame with columns: model, dataset, metric, slope, p_val

# Optional renaming
df = pd.read_csv('qa_ue_values.csv')
metric_map = {
    # "Comet-wmt22-comet-da": "COMET",
    # "XComet-XCOMET-XXL": "XCOMET",
    # "metricx-metricx-24-hybrid-large-v2p6": "MetricX"
      'MSP':'MSP', 'PPL' :'PPL', 'MTE' :'MTE',  'MCSE':'MCSE', 'MCNSE':'MCNSE', 'LSRL': 'LSRL'
}
df['metric'] = df['metric'].map(metric_map).fillna(df['metric'])

# Pivot to wide format with slope and p_val per metric
pivot = df.pivot_table(
    index=['model', 'dataset'],
    columns='metric',
    values=['slope', 'p_val'],
    aggfunc='first'
)

# Flatten MultiIndex columns
pivot.columns = [f"{stat}_{metric}" for stat, metric in pivot.columns]
pivot = pivot.reset_index()

# Define metric order
# metrics = ['COMET', 'MetricX', 'XCOMET']
metrics =['MSP', 'PPL', 'MTE',  'MCSE','MCNSE',  'LSRL']
columns = ['model', 'dataset']
for m in metrics:
    columns += [f'slope_{m}', f'p_val_{m}']
pivot = pivot.reindex(columns=columns)

# Build LaTeX table
lines = []

# Table header
lines.append("\\begin{tabular}{l" + "cc" * len(metrics) + "}")
lines.append("\\toprule")
lines.append("\\textbf{Dataset} & " + " & ".join([f"\\multicolumn{{2}}{{c}}{{{m}}}" for m in metrics]) + " \\\\")
lines.append(" & " + " & ".join(["slope & p-val"] * len(metrics)) + " \\\\")
lines.append("\\midrule")

# Model blocks without repeating model name in rows
for model, group in pivot.groupby("model"):
    lines.append("\\midrule")
    lines.append(f"\\multicolumn{{{1 + 2 * len(metrics)}}}{{c}}{{\\textbf{{{model}}}}} \\\\")
    lines.append("\\midrule")
    for _, row in group.iterrows():
        row_str = f"{format_dataset_name(row['dataset'])}"
        for m in metrics:
            s = row.get(f'slope_{m}', '')
            p = row.get(f'p_val_{m}', '')
            s_str = f"{s:.3f}" if pd.notna(s) else ""
            p_str = f"{p:.3f}" if pd.notna(p) else ""
            row_str += f" & {s_str} & {p_str}"
        lines.append(row_str + " \\\\")

lines.append("\\bottomrule")
lines.append("\\end{tabular}")

# Final LaTeX output
latex_code = "\n".join(lines)
print(latex_code)


\begin{tabular}{lcccccccccccc}
\toprule
\textbf{Dataset} & \multicolumn{2}{c}{MSP} & \multicolumn{2}{c}{PPL} & \multicolumn{2}{c}{MTE} & \multicolumn{2}{c}{MCSE} & \multicolumn{2}{c}{MCNSE} & \multicolumn{2}{c}{LSRL} \\
 & slope & p-val & slope & p-val & slope & p-val & slope & p-val & slope & p-val & slope & p-val \\
\midrule
\midrule
\multicolumn{13}{c}{\textbf{gemma}} \\
\midrule
GSM8K & 0.322 & 0.000 & -0.100 & 0.000 & -0.092 & 0.000 & 0.266 & 0.000 & 0.085 & 0.000 & 0.277 & 0.000 \\
\midrule
\multicolumn{13}{c}{\textbf{llama}} \\
\midrule
GSM8K & 0.418 & 0.000 & -0.109 & 0.000 & -0.098 & 0.000 & 0.273 & 0.000 & 0.096 & 0.000 & 0.219 & 0.000 \\
\bottomrule
\end{tabular}


In [39]:
import pandas as pd

# Assume df is your input DataFrame with columns: model, dataset, metric, slope, p_val

# Optional renaming
df = pd.read_csv('sum_values.csv')
metric_map = {
    "AlignScoreInputOutput": "Align Score",
    # "XComet-XCOMET-XXL": "XCOMET",
    # "metricx-metricx-24-hybrid-large-v2p6": "MetricX"
}


df['metric'] = df['metric'].map(metric_map).fillna(df['metric'])

# Pivot to wide format with slope and p_val per metric
pivot = df.pivot_table(
    index=['model', 'dataset'],
    columns='metric',
    values=['slope', 'p_val'],
    aggfunc='first'
)

# Flatten MultiIndex columns
pivot.columns = [f"{stat}_{metric}" for stat, metric in pivot.columns]
pivot = pivot.reset_index()

# Define metric order
metrics = ['Align Score']
columns = ['model', 'dataset']
for m in metrics:
    columns += [f'slope_{m}', f'p_val_{m}']
pivot = pivot.reindex(columns=columns)

# Build LaTeX table
lines = []

# Table header
lines.append("\\begin{tabular}{l" + "cc" * len(metrics) + "}")
lines.append("\\toprule")
lines.append("\\textbf{Dataset} & " + " & ".join([f"\\multicolumn{{2}}{{c}}{{{m}}}" for m in metrics]) + " \\\\")
lines.append(" & " + " & ".join(["\\textit{slope} & \\textit{p-val}"] * len(metrics)) + " \\\\")
lines.append("\\midrule")

# Model blocks without repeating model name in rows
for model, group in pivot.groupby("model"):
    lines.append(f"\\rowcolor[gray]{{0.9}} \\multicolumn{{{1 + 2 * len(metrics)}}}{{c}}{{\\textbf{{{model}}}}} \\\\")
    for _, row in group.iterrows():
        row_str = f"{format_dataset_name(row['dataset'])}"
        for m in metrics:
            s = row.get(f'slope_{m}', '')
            p = row.get(f'p_val_{m}', '')
            s_str = f"{s:.3f}" if pd.notna(s) else ""
            p_str = f"{p:.3f}" if pd.notna(p) else ""
            row_str += f" & {s_str} & {p_str}"
        lines.append(row_str + " \\\\")

lines.append("\\bottomrule")
lines.append("\\end{tabular}")

# Final LaTeX output
latex_code = "\n".join(lines)
print(latex_code)

\begin{tabular}{lcc}
\toprule
\textbf{Dataset} & \multicolumn{2}{c}{Align Score} \\
 & \textit{slope} & \textit{p-val} \\
\midrule
\rowcolor[gray]{0.9} \multicolumn{3}{c}{\textbf{gemma}} \\
XSUM & 0.126 & 0.000 \\
\rowcolor[gray]{0.9} \multicolumn{3}{c}{\textbf{llama}} \\
XSUM & 0.062 & 0.042 \\
\bottomrule
\end{tabular}


In [40]:

import pandas as pd

# Assume df is your input DataFrame with columns: model, dataset, metric, slope, p_val

# Optional renaming
df = pd.read_csv('qa_values.csv')
metric_map = {
    "Accuracy": "Accuracy",
    # "XComet-XCOMET-XXL": "XCOMET",
    # "metricx-metricx-24-hybrid-large-v2p6": "MetricX"
}


df['metric'] = df['metric'].map(metric_map).fillna(df['metric'])

# Pivot to wide format with slope and p_val per metric
pivot = df.pivot_table(
    index=['model', 'dataset'],
    columns='metric',
    values=['slope', 'p_val'],
    aggfunc='first'
)

# Flatten MultiIndex columns
pivot.columns = [f"{stat}_{metric}" for stat, metric in pivot.columns]
pivot = pivot.reset_index()

# Define metric order
metrics = ['Accuracy']
columns = ['model', 'dataset']
for m in metrics:
    columns += [f'slope_{m}', f'p_val_{m}']
pivot = pivot.reindex(columns=columns)

# Build LaTeX table
lines = []

# Table header
lines.append("\\begin{tabular}{l" + "cc" * len(metrics) + "}")
lines.append("\\toprule")
lines.append("\\textbf{Dataset} & " + " & ".join([f"\\multicolumn{{2}}{{c}}{{{m}}}" for m in metrics]) + " \\\\")
lines.append(" & " + " & ".join(["\\textit{slope} & \\textit{p-val}"] * len(metrics)) + " \\\\")
lines.append("\\midrule")

# Model blocks without repeating model name in rows
for model, group in pivot.groupby("model"):
    lines.append(f"\\rowcolor[gray]{{0.9}} \\multicolumn{{{1 + 2 * len(metrics)}}}{{c}}{{\\textbf{{{model}}}}} \\\\")
    for _, row in group.iterrows():
        row_str = f"{format_dataset_name(row['dataset'])}"
        for m in metrics:
            s = row.get(f'slope_{m}', '')
            p = row.get(f'p_val_{m}', '')
            s_str = f"{s:.3f}" if pd.notna(s) else ""
            p_str = f"{p:.3f}" if pd.notna(p) else ""
            row_str += f" & {s_str} & {p_str}"
        lines.append(row_str + " \\\\")

lines.append("\\bottomrule")
lines.append("\\end{tabular}")

# Final LaTeX output
latex_code = "\n".join(lines)
print(latex_code)


\begin{tabular}{lcc}
\toprule
\textbf{Dataset} & \multicolumn{2}{c}{Accuracy} \\
 & \textit{slope} & \textit{p-val} \\
\midrule
\rowcolor[gray]{0.9} \multicolumn{3}{c}{\textbf{gemma}} \\
GSM8K & -0.190 & 0.000 \\
\rowcolor[gray]{0.9} \multicolumn{3}{c}{\textbf{llama}} \\
GSM8K & -0.277 & 0.000 \\
\bottomrule
\end{tabular}
